In [23]:
import pandas as pd
import numpy as np
import random

In [24]:
df = pd.read_csv("matches.csv")
df.head()

,match_id,date,venue,team1,team2,stage,toss_winner,toss_decision,first_ings_score,first_ings_wkts,...,match_result,match_winner,wb_runs,wb_wickets,balls_left,player_of_the_match,top_scorer,highscore,best_bowling,best_bowling_figure
0,1,"March 22,2025","Eden Gardens, Kolkata",KKR,RCB,League,RCB,Bowl,174.0,8.0,...,completed,RCB,NaN,7,22.0,Krunal Pandya,Virat Kohli,59.0,Krunal Pandya,3--29
1,2,"March 23,2025","Rajiv Gandhi International Stadium, Hyderabad",SRH,RR,League,RR,Bowl,286.0,6.0,...,completed,SRH,44,NaN,0.0,Ishan Kishan,Ishan Kishan,106.0,Tushar Deshpande,3--44
2,3,"March 23,2025","MA Chidambaram Stadium, Chennai",CSK,MI,League,CSK,Bowl,155.0,9.0,...,completed,CSK,NaN,4,5.0,Noor Ahmad,Rachin Ravindra,65.0,Noor Ahmad,4--18
3,4,"March 24,2025","ACA-VDCA Cricket Stadium, Vishakhapatnam",DC,LSG,League,DC,Bowl,209.0,8.0,...,completed,DC,NaN,1,3.0,Ashutosh Sharma,Nicholas Pooran,75.0,Mitchell Starc,3--42
4,5,"March 25,2025","Narendra Modi Stadium, Ahmedabad",GT,PBKS,League,GT,Bowl,243.0,5.0,...,completed,PBKS,11,NaN,0.0,Shreyas Iyer,Shreyas Iyer,97.0,Sai Kishore,3--30


In [25]:
print(df.columns)

Index(['match_id', 'date', 'venue', 'team1', 'team2', 'stage', 'toss_winner',
       'toss_decision', 'first_ings_score', 'first_ings_wkts',
       'second_ings_score', 'second_ings_wkts', 'match_result', 'match_winner',
       'wb_runs', 'wb_wickets', 'balls_left', 'player_of_the_match',
       'top_scorer', 'highscore', 'best_bowling', 'best_bowling_figure'],
      dtype='object')


In [26]:
# Keep only required columns
df_clean = df[['team1', 'team2', 'match_winner']]

# Drop matches with no result
df_clean = df_clean.dropna(subset=['match_winner'])

# Rename for consistency
df_clean = df_clean.rename(columns={'match_winner': 'winner'})

df_clean.head()

,team1,team2,winner
0,KKR,RCB,RCB
1,SRH,RR,SRH
2,CSK,MI,CSK
3,DC,LSG,DC
4,GT,PBKS,PBKS


In [27]:
team_map = {
    "Royal Challengers Bengaluru": "RCB",
    "Mumbai Indians": "MI",
    "Chennai Super Kings": "CSK",
    "Kolkata Knight Riders": "KKR",
    "Sunrisers Hyderabad": "SRH",
    "Delhi Capitals": "DC",
    "Punjab Kings": "PBKS",
    "Rajasthan Royals": "RR",
    "Gujarat Titans": "GT",
    "Lucknow Super Giants": "LSG"
}

df_clean = df_clean.replace(team_map)

In [28]:
valid_teams = {"RCB","MI","CSK","KKR","SRH","DC","PBKS","RR","GT","LSG"}

df_clean = df_clean[
    df_clean['team1'].isin(valid_teams) &
    df_clean['team2'].isin(valid_teams)
]

In [29]:
matches_2025 = list(df_clean.itertuples(index=False, name=None))

print(matches_2025[:5])

[('KKR', 'RCB', 'RCB'), ('SRH', 'RR', 'SRH'), ('CSK', 'MI', 'CSK'), ('DC', 'LSG', 'DC'), ('GT', 'PBKS', 'PBKS')]


In [30]:
import numpy as np

def build_colley(matches, teams):
    n = len(teams)
    team_to_idx = {team: i for i, team in enumerate(teams)}

    C = np.zeros((n, n))
    b = np.zeros(n)

    games = np.zeros(n)
    wins = np.zeros(n)
    losses = np.zeros(n)

    for t1, t2, winner in matches:
        i = team_to_idx[t1]
        j = team_to_idx[t2]

        games[i] += 1
        games[j] += 1

        if winner == t1:
            wins[i] += 1
            losses[j] += 1
        else:
            wins[j] += 1
            losses[i] += 1

        C[i][j] -= 1
        C[j][i] -= 1

    for i in range(n):
        C[i][i] = 2 + games[i]
        b[i] = 1 + (wins[i] - losses[i]) / 2

    ratings = np.linalg.solve(C, b)

    return ratings, team_to_idx

In [31]:
matches_2026 = [
    ("RCB", "SRH", "RCB"),
    ("MI", "KKR", "MI"),
    ("DC", "LSG", "DC"),
    ("CSK", "RR", "RR"),
    ("PBKS", "GT", "PBKS"),
    ("RR", "GT", "RR"),
    ("RCB", "CSK", "RCB"),
    ("MI", "DC", "DC"),
    ("KKR", "SRH", "SRH"),
    ("SRH", "LSG", "LSG"),
    ("CSK", "PBKS", "PBKS"),

]

In [32]:
teams = ["RCB","MI","CSK","KKR","SRH","DC","PBKS","RR","GT","LSG"]

r_2025, _ = build_colley(matches_2025, teams)
r_2026, team_to_idx = build_colley(matches_2026, teams)

final_ratings = 0.7 * r_2025 + 0.3 * r_2026

In [33]:
for team, idx in team_to_idx.items():
    print(team, round(final_ratings[idx], 3))

RCB 0.658
MI 0.505
CSK 0.318
KKR 0.352
SRH 0.442
DC 0.652
PBKS 0.661
RR 0.429
GT 0.502
LSG 0.483


In [34]:
fixtures = [
    ("RCB","MI"),
    ("RR","PBKS"),
    ("LSG","KKR"),
    ("CSK","DC"),
    ("GT","SRH"),
    ("RCB","KKR"),
    ("RR","DC"),
    ("KKR","RR"),
    ("PBKS","CSK"),
    ("LSG","MI"),
    ("SRH","CSK"),
    ("RCB","MI"),
    ("GT","RCB"),
    ("SRH","KKR"),
    ("PBKS","GT"),
    ("SRH","LSG"),
    ("MI","RCB"),
    ("DC","GT"),
    ("RR","PBKS"),
    ("KKR","CSK"),

]

In [35]:
HOME_ADV = 0.05
EPS = 1e-6

def predict_season(fixtures, ratings, team_to_idx):
    results = []
    points = {team: 0 for team in team_to_idx}

    for home, away in fixtures:
        r_home = ratings[team_to_idx[home]] + HOME_ADV
        r_away = ratings[team_to_idx[away]]


        if abs(r_home - r_away) < EPS:
            winner = random.choice([home, away])
        else:
            winner = home if r_home > r_away else away

        results.append((home, away, winner))
        points[winner] += 2

    return results, points

In [36]:
results, points = predict_season(fixtures, final_ratings, team_to_idx)

In [37]:
for r in results:
    print(f"{r[0]} vs {r[1]} → {r[2]}")

RCB vs MI → RCB
RR vs PBKS → PBKS
LSG vs KKR → LSG
CSK vs DC → DC
GT vs SRH → GT
RCB vs KKR → RCB
RR vs DC → DC
KKR vs RR → RR
PBKS vs CSK → PBKS
LSG vs MI → LSG
SRH vs CSK → SRH
RCB vs MI → RCB
GT vs RCB → RCB
SRH vs KKR → SRH
PBKS vs GT → PBKS
SRH vs LSG → SRH
MI vs RCB → RCB
DC vs GT → DC
RR vs PBKS → PBKS
KKR vs CSK → KKR


In [38]:
sorted_points = sorted(points.items(), key=lambda x: x[1], reverse=True)

for team, pts in sorted_points:
    print(team, pts)

RCB 10
PBKS 8
SRH 6
DC 6
LSG 4
KKR 2
RR 2
GT 2
MI 0
CSK 0


In [39]:
import numpy as np
import pandas as pd
import random

# -------------------------------
# COLLEY FUNCTION
# -------------------------------
def build_colley(matches, teams):
    n = len(teams)
    team_to_idx = {team: i for i, team in enumerate(teams)}

    C = np.zeros((n, n))
    b = np.zeros(n)

    games = np.zeros(n)
    wins = np.zeros(n)
    losses = np.zeros(n)

    for t1, t2, winner in matches:
        i = team_to_idx[t1]
        j = team_to_idx[t2]

        games[i] += 1
        games[j] += 1

        if winner == t1:
            wins[i] += 1
            losses[j] += 1
        else:
            wins[j] += 1
            losses[i] += 1

        C[i][j] -= 1
        C[j][i] -= 1

    for i in range(n):
        C[i][i] = 2 + games[i]
        b[i] = 1 + (wins[i] - losses[i]) / 2

    ratings = np.linalg.solve(C, b)
    return ratings, team_to_idx, C, b


# -------------------------------
# TEAMS
# -------------------------------
teams = ["RCB","MI","CSK","KKR","SRH","DC","PBKS","RR","GT","LSG"]

# -------------------------------
# CLEAN 2025 DATA
# -------------------------------
df_clean = df[['team1', 'team2', 'match_winner']].dropna()
df_clean = df_clean.rename(columns={'match_winner': 'winner'})

valid_teams = set(teams)
df_clean = df_clean[
    df_clean['team1'].isin(valid_teams) &
    df_clean['team2'].isin(valid_teams) &
    df_clean['winner'].isin(valid_teams)
]

matches_2025 = list(df_clean.itertuples(index=False, name=None))


# -------------------------------
# 2026 FIRST 11 MATCHES
# -------------------------------
matches_2026 = [
    ("RCB","SRH","RCB"),
    ("MI","KKR","MI"),
    ("DC","LSG","DC"),
    ("CSK","RR","RR"),
    ("PBKS","GT","PBKS"),
    ("RR","GT","RR"),
    ("RCB","CSK","RCB"),
    ("MI","DC","DC"),
    ("KKR","SRH","SRH"),
    ("SRH","LSG","LSG"),
    ("CSK","PBKS","PBKS"),
]

# -------------------------------
# BUILD INITIAL COLLEY
# -------------------------------
r_2025, _, _, _ = build_colley(matches_2025, teams)
r_2026, team_to_idx, _, _ = build_colley(matches_2026, teams)

# Hybrid ratings
final_ratings = 0.7 * r_2025 + 0.3 * r_2026


# -------------------------------
# FIXTURES (MATCH 12–70)
# -------------------------------
fixtures = [
    ("KKR","PBKS"), ("RR","MI"), ("DC","GT"), ("KKR","LSG"), ("RR","RCB"),
    ("PBKS","SRH"), ("CSK","DC"), ("LSG","GT"), ("MI","RCB"), ("SRH","RR"),
    ("CSK","KKR"), ("RCB","LSG"), ("MI","PBKS"), ("GT","KKR"), ("RCB","DC"),
    ("SRH","CSK"), ("KKR","RR"), ("PBKS","LSG"), ("GT","MI"), ("SRH","DC"),
    ("LSG","RR"), ("MI","CSK"), ("RCB","GT"), ("DC","PBKS"), ("RR","SRH"),
    ("GT","CSK"), ("LSG","KKR"), ("DC","RCB"), ("PBKS","RR"), ("MI","SRH"),
    ("GT","RCB"), ("RR","DC"), ("CSK","MI"), ("SRH","KKR"), ("GT","PBKS"),
    ("MI","LSG"), ("DC","CSK"), ("SRH","PBKS"), ("LSG","RCB"), ("DC","KKR"),
    ("RR","GT"), ("CSK","LSG"), ("RCB","MI"), ("PBKS","DC"), ("GT","SRH"),
    ("RCB","KKR"), ("PBKS","MI"), ("LSG","CSK"), ("KKR","GT"), ("PBKS","RCB"),
    ("DC","RR"), ("CSK","SRH"), ("RR","LSG"), ("KKR","MI"), ("CSK","GT"),
    ("SRH","RCB"), ("LSG","PBKS"), ("MI","RR"), ("KKR","DC")
]

# -------------------------------
# SIMULATION
# -------------------------------
HOME_ADV = 0.05
TOSS_ADV = 0.03

results = []

for home, away in fixtures:
    toss_winner = random.choice([home, away])

    r_home = final_ratings[team_to_idx[home]] + HOME_ADV
    r_away = final_ratings[team_to_idx[away]]

    if toss_winner == home:
        r_home += TOSS_ADV
    else:
        r_away += TOSS_ADV


    if abs(r_home - r_away) < 1e-6:
        winner = random.choice([home, away])
    else:
        winner = home if r_home > r_away else away

    results.append((home, away, winner))


# -------------------------------
# BUILD FULL SEASON MATCH LIST
# -------------------------------
full_matches = []

for t1, t2, winner in matches_2026:
    full_matches.append((t1, t2, winner))

for t1, t2, winner in results:
    full_matches.append((t1, t2, winner))


# -------------------------------
# FINAL COLLEY (FULL SEASON)
# -------------------------------
r_full, team_to_idx, C_full, b_full = build_colley(full_matches, teams)


# -------------------------------
# OUTPUTS
# -------------------------------
print("=== FINAL SEASON COLLEY MATRIX ===")
print(C_full)

print("\n=== FINAL b VECTOR ===")
print(b_full)

print("\n=== FINAL COLLEY RATINGS ===")
for team, idx in team_to_idx.items():
    print(team, round(r_full[idx], 3))


# -------------------------------
# FINAL STANDINGS
# -------------------------------
points = {team: 0 for team in teams}

for _, _, winner in full_matches:
    points[winner] += 2

table = sorted(points.items(), key=lambda x: x[1], reverse=True)

print("\n=== FINAL STANDINGS ===")
for i, (team, pts) in enumerate(table, 1):
    print(f"{i}. {team} - {pts}")

print("\n=== TOP 4 ===")
for team, pts in table[:4]:
    print(team, pts)

=== FINAL SEASON COLLEY MATRIX ===
[[16. -2. -1. -1. -2. -2. -1. -1. -2. -2.]
 [-2. 16. -2. -2. -1. -1. -2. -2. -1. -1.]
 [-1. -2. 16. -1. -2. -2. -1. -1. -2. -2.]
 [-1. -2. -1. 16. -2. -2. -1. -1. -2. -2.]
 [-2. -1. -2. -2. 16. -1. -2. -2. -1. -1.]
 [-2. -1. -2. -2. -1. 16. -2. -2. -1. -1.]
 [-1. -2. -1. -1. -2. -2. 16. -1. -2. -2.]
 [-1. -2. -1. -1. -2. -2. -1. 16. -2. -2.]
 [-2. -1. -2. -2. -1. -1. -2. -2. 16. -1.]
 [-2. -1. -2. -2. -1. -1. -2. -2. -1. 16.]]

=== FINAL b VECTOR ===
[ 6.  2. -6. -4. -1.  6.  7. -3.  1.  2.]

=== FINAL COLLEY RATINGS ===
RCB 0.807
MI 0.545
CSK 0.102
KKR 0.219
SRH 0.369
DC 0.781
PBKS 0.866
RR 0.278
GT 0.487
LSG 0.545

=== FINAL STANDINGS ===
1. PBKS - 26
2. RCB - 24
3. DC - 24
4. MI - 16
5. LSG - 16
6. GT - 14
7. SRH - 10
8. RR - 6
9. KKR - 4
10. CSK - 0

=== TOP 4 ===
PBKS 26
RCB 24
DC 24
MI 16
